# Positional Encoding

语言是离散序列数据，对于 “喝蜂蜜” 和 “喝蜜蜂”， 虽然其词相同，但由于词之间顺序不同，导致不同的语义。所以语言建模中，序列性对语义的决定是至关重要的。语言模型的输入信息源分两种：

1. embedding：词嵌入
2. position：位置信息，可以是标量、向量或矩阵，等多种信息。


## 分析RNN 模型和Attention 模型

**RNN 模型**

$ h_t = f(h_{t-1} + x_t) $

其中通过上个时刻的 隐状态 $h_{t-1}$ 与当前状态 $x_t$ 进行 变换算子$f(\cdot)$计算得到 当前时刻的 隐状态 $h_t$, 在 RNN 模型中是不需要位置编码的，因为相邻关系就能表示时序关系。

**Attention 模型**

$ h_{1:t} = \texttt{Attn}(x_{1:t}) $

其中，$\texttt{Attn}$ 为注意力算子，多个token的词嵌入，经过序列建模，得到多个token的注意力特征向量。那么为什么 Attention 机制需要 位置编码？

# Attention 示例

In [5]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math
torch.manual_seed(42)

In [9]:
class Model(nn.Module):
    def __init__(self, vocab_size = 100, d_model = 8):
        super().__init__()
        self.vocab_size = vocab_size
        self.d_model = d_model
        self.embd = nn.Embedding(self.vocab_size, self.d_model) 
        self.wq = nn.Linear(d_model, d_model, bias=False)
        self.wk = nn.Linear(d_model, d_model, bias=False)
        self.wv = nn.Linear(d_model, d_model, bias=False)
        self.wo = nn.Linear(d_model, d_model, bias=False)
        
    def forward(self, x):
        X = self.embd(x)
        Q,K,V = self.wq(X), self.wk(X), self.wv(X)
        O = F.softmax(Q@K.t() / math.sqrt(self.d_model), dim = -1) @  V
        return O
vocab_size = 100
d = 8
model = Model(vocab_size=vocab_size, d_model = d)

tensor([[-0.2297, -0.3818, -0.1324,  0.1058,  0.2297, -0.0741, -0.1921, -0.4140],
        [-0.1984, -0.3775, -0.1363,  0.0463,  0.2340, -0.0091, -0.1384, -0.3292],
        [-0.2405, -0.3854, -0.1262,  0.1306,  0.2324, -0.0921, -0.1935, -0.4343]],
       grad_fn=<MmBackward0>)

In [13]:
# 调整token序列的顺序，发现各个 token 的注意力特征同样置换了顺序
# 对注意力特征进行合并，两个序列的 sum 语义向量是一样的。

input_ids = torch.tensor([3,8,4], dtype = torch.long)
y = model(input_ids)
print('original embd:', y)
print('original sum(embd)', y.sum(dim = 0))

input_ids = torch.tensor([8,3,4], dtype = torch.long)
y = model(input_ids)
print('shift embd:', y)
print('shift sum(embd)', y.sum(dim = 0))

original embd: tensor([[-0.2297, -0.3818, -0.1324,  0.1058,  0.2297, -0.0741, -0.1921, -0.4140],
        [-0.1984, -0.3775, -0.1363,  0.0463,  0.2340, -0.0091, -0.1384, -0.3292],
        [-0.2405, -0.3854, -0.1262,  0.1306,  0.2324, -0.0921, -0.1935, -0.4343]],
       grad_fn=<MmBackward0>)
original sum(embd) tensor([-0.6685, -1.1448, -0.3950,  0.2827,  0.6961, -0.1753, -0.5240, -1.1775],
       grad_fn=<SumBackward1>)
shift embd: tensor([[-0.1984, -0.3775, -0.1363,  0.0463,  0.2340, -0.0091, -0.1384, -0.3292],
        [-0.2297, -0.3818, -0.1324,  0.1058,  0.2297, -0.0741, -0.1921, -0.4140],
        [-0.2405, -0.3854, -0.1262,  0.1306,  0.2324, -0.0921, -0.1935, -0.4343]],
       grad_fn=<MmBackward0>)
shift sum(embd) tensor([-0.6685, -1.1448, -0.3950,  0.2827,  0.6961, -0.1753, -0.5240, -1.1775],
       grad_fn=<SumBackward1>)


## 标量位置信息

In [22]:
class ModelScalar(nn.Module):
    def __init__(self, vocab_size = 100, d_model = 8):
        super().__init__()
        self.vocab_size = vocab_size
        self.d_model = d_model
        self.embd = nn.Embedding(self.vocab_size, self.d_model) 
        self.wq = nn.Linear(d_model, d_model, bias=False)
        self.wk = nn.Linear(d_model, d_model, bias=False)
        self.wv = nn.Linear(d_model, d_model, bias=False)
        self.wo = nn.Linear(d_model, d_model, bias=False)
        
    def forward(self, x):
        seq_len = x.shape[0]
        pe = torch.arange(seq_len).unsqueeze(1) # 标量位置信息
        print(pe)
        X = self.embd(x) 
        X += pe # 输入嵌入位置信息
        Q,K,V = self.wq(X), self.wk(X), self.wv(X)
        O = F.softmax(Q@K.t() / math.sqrt(self.d_model), dim = -1) @  V
        return O
        
model = ModelScalar(vocab_size=vocab_size, d_model = d)

In [39]:
input_ids = torch.tensor([3,8], dtype = torch.long)
seq_len = input_ids.shape[0]
pe = torch.arange(seq_len).unsqueeze(1) # 标量位置信息
print(pe)
print(pe.shape)
X = model.embd(input_ids)
print(X.shape)
print(X)
X_pe = X + pe # 输入嵌入位置信息
print(X_pe)
print(X_pe - X)

tensor([[0],
        [1]])
torch.Size([2, 1])
torch.Size([2, 8])
tensor([[ 0.2839,  0.5940,  0.3153,  0.2705,  0.9633, -1.2189, -0.0693,  0.0610],
        [-0.4278, -0.5278, -1.7987, -0.0768,  0.7816,  0.1391, -1.7340,  0.7391]],
       grad_fn=<EmbeddingBackward0>)
tensor([[ 0.2839,  0.5940,  0.3153,  0.2705,  0.9633, -1.2189, -0.0693,  0.0610],
        [ 0.5722,  0.4722, -0.7987,  0.9232,  1.7816,  1.1391, -0.7340,  1.7391]],
       grad_fn=<AddBackward0>)
tensor([[0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000]],
       grad_fn=<SubBackward0>)


In [40]:
input_ids = torch.tensor([3,8,4], dtype = torch.long)
y = model(input_ids)
print('original embd:', y)
print('original sum(embd)', y.sum(dim = 0))

input_ids = torch.tensor([8,3,4], dtype = torch.long)
y = model(input_ids)
print('shift embd:', y)
print('shift sum(embd)', y.sum(dim = 0))

tensor([[0],
        [1],
        [2]])
original embd: tensor([[-0.6592,  0.5683,  0.5001,  0.3393,  0.5457,  0.9358, -0.3715,  0.1487],
        [-0.4912,  0.5642,  0.3807,  0.3041,  0.5870,  0.8822, -0.3322,  0.1588],
        [-0.5699,  0.6903,  0.3912,  0.4157,  0.1555,  0.7193, -0.0514,  0.0084]],
       grad_fn=<MmBackward0>)
original sum(embd) tensor([-1.7202,  1.8228,  1.2720,  1.0591,  1.2882,  2.5372, -0.7551,  0.3159],
       grad_fn=<SumBackward1>)
tensor([[0],
        [1],
        [2]])
shift embd: tensor([[-0.7036,  0.4907,  0.5772,  0.3120,  0.9702,  1.2256, -0.6883,  0.2543],
        [-0.6778,  0.6405,  0.4412,  0.3336, -0.1718,  0.4067,  0.1156,  0.0198],
        [-0.6986,  0.6738,  0.4198,  0.3289, -0.5370,  0.1380,  0.3583, -0.0430]],
       grad_fn=<MmBackward0>)
shift sum(embd) tensor([-2.0800,  1.8050,  1.4382,  0.9745,  0.2614,  1.7702, -0.2144,  0.2311],
       grad_fn=<SumBackward1>)


上述代码实现中，引入标量位置信息，对输入嵌入位置，可见最终的注意力特征及完整文本语义表示都有变化。

## 注意分数计算

给定位置m，n向量，没有位置信息时：

$$
\begin{align}
& X_mW_Q \times (X_nW_K)^T 
\end{align}
$$

有位置信息

$$
\begin{align}
&(X_m+P_m)W_Q \times ((X_n+P_n)W_K)^T \\
&=(X_mW_Q+P_mW_Q) \times (W_K^TX_n^T+W_K^TP_n^T)\\
&=X_mW_QW_K^TX_n^T + X_mW_Q W_K^TP_n^T +  P_mW_QW_K^TX_n^T + P_mW_QW_K^TP_n^T
\end{align}
$$

其中, $P_mW_QW_K^TP_n^T = (m*I)W_QW_K(n*I) = mn(W_QW_K)$ 可见该项中的两个位置信息产生交互

## Transformer位置编码

根据上述标量位置信息，已能区分调整序列顺序前后的特征向量表示了。缺点在于：

1. 每个维度位置信息一样，表示单一
2. 在注意分数形式下，无法体现位置相对性。

绝对位置编码为

$$
PE(n) = [\sin(n\theta_0),\cos(n\theta_0), \ldots, \sin(n\theta_{d/2-1}),\cos(n\theta_{d/2-1})]
$$

其中，PE(n)表示 位置$n$ 的绝对位置编码向量， $\theta_i = \frac{1}{b^{i*2/d}}$, $b=10000.0$

In [42]:
a = torch.arange(10)
print(a[0::2])
print(a[1::2])

tensor([0, 2, 4, 6, 8])
tensor([1, 3, 5, 7, 9])


In [49]:
class PositionalEncoding(nn.Module):
    def __init__(self, max_len = 100, d_model = 8, base = 10000.0):
        super().__init__()
        d = d_model // 2
        i = torch.arange(d)
        theta = base ** ( - i * 2 / d )
        L = torch.arange(max_len)

        m_theta = torch.outer(L, theta)
        sin = m_theta.sin()
        cos = m_theta.cos()

        self.PE = torch.zeros(max_len, d_model)
        self.PE[:, 0::2] = sin
        self.PE[:, 1::2] = cos
        
        
    def forward(self, x):
        seq_len = x.shape[0]
        return self.PE[:seq_len, :]
 
pe = PositionalEncoding(max_len = 100, d_model = d, base = 10000.0)

input_ids = torch.tensor([3,8,4], dtype = torch.long)
pe(input_ids)

tensor([[ 0.0000e+00,  1.0000e+00,  0.0000e+00,  1.0000e+00,  0.0000e+00,
          1.0000e+00,  0.0000e+00,  1.0000e+00],
        [ 8.4147e-01,  5.4030e-01,  9.9998e-03,  9.9995e-01,  1.0000e-04,
          1.0000e+00,  1.0000e-06,  1.0000e+00],
        [ 9.0930e-01, -4.1615e-01,  1.9999e-02,  9.9980e-01,  2.0000e-04,
          1.0000e+00,  2.0000e-06,  1.0000e+00]])

In [50]:
class ModelPE(nn.Module):
    def __init__(self, vocab_size = 100, d_model = 8, max_len = 100, base = 10000.0):
        super().__init__()
        self.vocab_size = vocab_size
        self.d_model = d_model
        self.embd = nn.Embedding(self.vocab_size, self.d_model) 
        self.PE = PositionalEncoding(max_len = max_len, d_model = d_model, base = base)
        
        self.wq = nn.Linear(d_model, d_model, bias=False)
        self.wk = nn.Linear(d_model, d_model, bias=False)
        self.wv = nn.Linear(d_model, d_model, bias=False)
        self.wo = nn.Linear(d_model, d_model, bias=False)
        
    def forward(self, x):
        seq_len = x.shape[0]
        X = self.embd(x) + self.PE(x) # addpe
        Q,K,V = self.wq(X), self.wk(X), self.wv(X)
        O = F.softmax(Q@K.t() / math.sqrt(self.d_model), dim = -1) @  V
        return O
        
model = ModelPE(vocab_size=100, d_model = d, max_len = 100,base = 10000.0)

input_ids = torch.tensor([3,8,4], dtype = torch.long)
model(input_ids)

tensor([[-0.2684, -0.0155, -0.2947,  0.0571,  0.1456,  0.3240, -0.1990,  0.5573],
        [-0.0927, -0.0467, -0.4740,  0.0744,  0.3124,  0.3451, -0.0632,  0.7025],
        [-0.3577, -0.0025, -0.2010,  0.1511,  0.2134,  0.3222, -0.1760,  0.6092]],
       grad_fn=<MmBackward0>)

## 绝对位置编码分析



1. 为什么每个维度的角度需要有变化
3. 为什么以 sin-cos 为组
4. 为什么base要设置成 10000.0
5. 位置编码的远程衰减性
6. 可学习位置编码
7. 无位置编码是否可行